In [ ]:
import torch
import cupy as cp
import triton
import triton.language as tl

In [ ]:
@triton.jit
def embedding_forward_kernel(
        embeddings_ptr,
        indices_ptr,
        output_ptr,
        n_elements,
        embedding_dim: tl.constexpr,
        BLOCK_SIZE_M: tl.constexpr,     # how many embeds to load at a time
        BLOCK_SIZE_N: tl.constexpr,     # block along the embed vector we want to load
        DTYPE_FLAG: tl.constexpr
):


    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)


    # cast pointers
    indices_ptr = tl.cast(indices_ptr, tl.pointer_type(tl.int32))
    embeddings_ptr = tl.cast(embeddings_ptr, tl.pointer_type(tl.float32 if DTYPE_FLAG == 0 else tl.float16))
    output_ptr = tl.cast(output_ptr, tl.pointer_type(tl.float32 if DTYPE_FLAG == 0 else tl.float16))


    start_m = pid_m * BLOCK_SIZE_M
    offsets_m = start_m + tl.arange(0, BLOCK_SIZE_M)
    mask_m = (offsets_m < n_elements)
    indices = tl.load(indices_ptr + offsets_m, mask=mask_m, other=0)


    start_n = pid_n * BLOCK_SIZE_N
    offsets_n = start_n + tl.arange(0, BLOCK_SIZE_N)
    mask_n = (offsets_n < embedding_dim)


    combined_mask = mask_m[:, None] & mask_n[None, :]


    # get the offsets now (BLOCK_SIZE_M, 1) + (1, BLOCK_SIZE_N)
    embeddings_offsets = indices[:, None] * embedding_dim + offsets_n[None, :]
    embeddings = tl.load(embeddings_ptr + embeddings_offsets,
                         mask=combined_mask,
                         other=0.0)


    output_offsets = offsets_m[:, None] * embedding_dim + offsets_n[None, :]
    tl.store(output_ptr + output_offsets, embeddings, mask=combined_mask)



In [ ]:
def fused_embedding_forward(embeddings, indices, use_dlpack=True):


    indices = indices.astype("int32", copy=False)


    assert indices > 0
    assert indices < embeddings.shape[0]


    if use_dlpack:
        original_indices_shape = indices.shape  # B x S
        indices = indices.reshape(-1)   # B*S


        embeddings = torch.utils.dlpack.from_dlpack(embeddings)
        indices = torch.utils.dlpack.from_dlpack(indices)


        if not embeddings.is_contiguous():
            embeddings = embeddings.contiguous()
        if not indices.is_contiguous():
            indices = indices.contiguous()


        n_elements = indices.shape[0]
        embed_dim = embeddings.shape[1]


        output = torch.empty(
            indices.shape[0],
            embeddings.shape[1],
            device=indices.device,
            dtype=embeddings.dtype,
        )


        # embed_dim << vocab_size
        BLOCK_SIZE_M = triton.next_power_of_2(min(128, embed_dim))
        BLOCK_SIZE_N = triton.next_power_of_2(min(128, embed_dim))


        grid = (
            torch.cdiv(n_elemnts, BLOCK_SIZE_M),
            torch.cdiv(embed_dim, BLOCK_SIZE_N)
        )


        embedding_forward_kernel[grid](
            embeddings_ptr=embeddings,
            indices_ptr=indices,
            output_ptr=output,
            n_elements=n_elements,
            embedding_dim=embed_dim,
            BLOCK_SIZE_M=BLOCK_SIZE_M,
            BLOCK_SIZE_N=BLOCK_SIZE_N,
            DTYPE_FLAG=0 if embeddings.dtype == torch.float32 else 1,
        )


        output.contiguous()
        return cp.from_dlpack(output.reshape(*original_indices_shape, -1))


In [ ]:
@triton.jit
def embedding_backward_kernel(
        grad_output_ptr,
        grad_weight_ptr,
        indices_ptr,
        n_elements,
        embedding_dim: tl.constexpr,
        BLOCK_SIZE_M: tl.constexpr,
        BLOCK_SIZE_N: tl.constexpr,
        DTYPE_FLAG: tl.constexpr,       # 0 for float32, 1 for float16
):

    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)


    # cast pointers
    indices_ptr = tl.cast(indices_ptr, tl.pointer_type(tl.int32))
    grad_weight_ptr = tl.cast(grad_weight_ptr, tl.pointer_type(tl.float32 if DTYPE_FLAG == 0 else tl.float16))
    grad_output_ptr = tl.cast(grad_output_ptr, tl.pointer_type(tl.float32 if DTYPE_FLAG == 0 else tl.float16))


    # get the indexes of what embeddings we are loading
    start_m = pid_m * BLOCK_SIZE_M
    offsets_m = start_m + tl.arange(0, BLOCK_SIZE_M)
    mask_m = offsets_m < n_elements
    indices = tl.load(indices_ptr + offsets_m, mask=mask_m, other=0)


    # get the indexes of what part of those embeddings we are loading
    start_n = pid_n * BLOCK_SIZE_N
    offsets_n = start_n + tl.arange(0, BLOCK_SIZE_N)
    mask_n = offsets_n < embedding_dim


    grad_output = tl.load(
        grad_output_ptr + offsets_m[:, None] * embedding_dim + offsets_n[None, :],
        mask=mask_m[:, None] & mask_n[None, :],
        other=0.0,
    )


    grad_weight_offsets = indices[:, None] * embedding_dim + offsets_n[None, :]


    tl.atomic_add(
        grad_weight_ptr + grad_weight_offsets,
        grad_output,
        mask=mask_m[:, None] & mask_n[None, :],
    )



In [ ]:
def fused_embedding_backward(grad_output, embeddings, indices, use_dlpack=True):

    # only support int32 indexes
    indices = indices.astype("int32", copy = False)

    assert (indices >= 0).all(), f"negative indices found: {indices.min()}"
    assert (indices < embeddings.shape[0]).all(), f"index out of bounds: max={indices.max()}, vocab={embeddings.shape[0]}"

    if use_dlpack:
        # flatten
        indices = indices.reshape(-1)

        grad_output = torch.utils.dlpack.from_dlpack(grad_output)
        embeddings = torch.utils.dlpack.from_dlpack(embeddings)
        indices = torch.utils.dlpack.from_dlpack(indices)

        if not grad_output.is_contiguous():
            grad_output = grad_output.contiguous()
        if not embeddings.is_contiguous():
            embeddings = embeddings.contiguous()
        if not indices.is_contiguous():
            indices = indices.contiguous()

        grad_weight = torch.zeros_like(embeddings)

        n_elements = indices.shape[0]
        embed_dim = embeddings.shape[1]

        BLOCK_SIZE_M = triton.next_power_of_2(min(128, embed_dim))
        BLOCK_SIZE_N = triton.next_power_of_2(min(128, embed_dim))

        grid = (
            triton.cdiv(n_elements, BLOCK_SIZE_M),
            triton.cdiv(embed_dim, BLOCK_SIZE_N),
        )

        embedding_backward_kernel[grid](
            grad_output,
            grad_weight,
            indices,
            n_elements,
            embedding_dim=embedding_dim,
            BLOCK_SIZE_M=BLOCK_SIZE_M,
            BLOCK_SIZE_N=BLOCK_SIZE_N,
            DTYPE_FLAG=0 if embeddings.dtype == torch.float32 else 1,
        )

        return cp.from_dlpack(grad_weight)

    else:
        # flatten
        indices = indices.reshape(-1)

        if not grad_output.flags.c_contiguous():
            grad_output = cp.ascontiguousarray(grad_output)
        if not embeddings.flags.c_contiguous:
            embeddings = cp.ascontiguousarray(embeddings)
        if not indices.flags.c_contiguous:
            indices = cp.ascontiguousarray(indices)

        grad_weight = cp.zeros_like(embeddings)

        n_elements = indices.shape[0]
        embed_dim = embeddings.shape[1]

        BLOCK_SIZE_M = triton.next_power_of_2(min(128, embed_dim))
        BLOCK_SIZE_N = triton.next_power_of_2(min(128, embed_dim))

        grid = (
            triton.cdiv(n_elements, BLOCK_SIZE_M),
            triton.cdiv(embed_dim, BLOCK_SIZE_N),
        )

        embedding_backward_kernel[grid](
            grad_output.data.ptr,
            grad_weight.data.ptr,
            indices.data.ptr,
            n_elements,
            embedding_dim=embedding_dim,
            BLOCK_SIZE_M=BLOCK_SIZE_M,
            BLOCK_SIZE_N=BLOCK_SIZE_N,
            DTYPE_FLAG=0 if embeddings.dtype == cp.float32 else 1,
        )

        return grad_weight

In [ ]:
if __name__ == "__main__":

    from torch.nn import Embedding

    def test_fused_embedding():
        torch.manual_seed(0)

        # config
        vocab_size = 10
        embed_dim = 16
        batch_size = 4
        seq_len = 3

        embedding = Embedding(vocab_size, embed_dim, _weight=torch.randn(vocab_size, embed_dim, device="cuda", dtype=torch.float16))
        indices = torch.randint(0, vocab_size, (batch_size, seq_len), device="cuda", dtype=torch.int32)
        embeddings_cp = cp.from_dlpack(torch.utils.dlpack.to_dlpack(embedding.weight))
        indices_cp = cp.from_dlpack(torch.utils.dlpack.to_dlpack(indices))
        out_cp = fused_embedding_forward(embeddings_cp, indices_cp, use_dlpack=False)
        out_torch = embedding(indices)
        torch.testing.assert_close(torch.tensor(out_cp.get(), device="cuda", dtype=out_torch.dtype), out_torch, atol=1e-3, rtol=1e-3)
        print("Forward pass matches PyTorch embedding output")

        grad_output = torch.randn_like(out_torch)
        grad_output_cp = cp.from_dlpack(torch.utils.dlpack.to_dlpack(grad_output))
        grad_weight_cp = fused_embedding_backward(grad_output_cp, embeddings_cp, indices_cp, use_dlpack=True)
        grad_weight_torch = torch.zeros_like(embedding.weight)
        grad_weight_torch.index_add_(0, indices.view(-1), grad_output.view(-1, embed_dim))
        torch.testing.assert_close(torch.tensor(grad_weight_cp.get(), device="cuda", dtype=grad_weight_torch.dtype), grad_weight_torch, atol=1e-3, rtol=1e-3)
        print("Backward pass matches PyTorch index_add_ gradient")

    test_fused_embedding()
